# ChestMNIST DINOv2 Skeleton Demo

This notebook checks the shared data, model, training, evaluation, and XAI interfaces on ChestMNIST.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

requirements_file = project_root / "requirements.txt"
print("Python:", sys.executable)
print("Project root:", project_root)
print("Requirements:", requirements_file)

If the imports below fail with `ModuleNotFoundError`, uncomment and run the next cell once in this notebook kernel, then restart the kernel.

In [ ]:
# %pip install -r "{requirements_file}"

In [ ]:
import pandas as pd
import torch

from src.data import get_small_data
from src.model import get_dino_model, get_probs, count_trainable_params, count_total_params
from src.train import train_model
from src.eval import classification_metrics
from src.xai import make_heatmap, show_heatmap

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
epochs = 20
batch_size = 20
learning_rate = 1e-3

# None means use the full split. Set small integers for quick sanity checks.
n_train = None
n_val = None

In [ ]:
train_loader, val_loader, class_names = get_small_data(
    n_train=n_train,
    n_val=n_val,
    batch_size=batch_size,
)

batch = next(iter(train_loader))
print("train batches:", len(train_loader))
print("val batches:", len(val_loader))
print("images:", batch["images"].shape)
print("labels:", batch["labels"].shape)
print("ids:", batch["ids"][:3])
print("classes:", class_names)

In [ ]:
model = get_dino_model(num_classes=len(class_names)).to(device)

print("total params:", f"{count_total_params(model):,}")
print("trainable params:", f"{count_trainable_params(model):,}")

In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=epochs,
    lr=learning_rate,
)

pd.DataFrame(history).drop(columns=["val_auc_per_class"])

In [ ]:
probs, labels, ids = get_probs(model, val_loader, device)

print("probs:", probs.shape)
print("labels:", labels.shape)
print("ids:", len(ids))

In [ ]:
metrics = classification_metrics(probs, labels)
print(metrics)

In [ ]:
batch = next(iter(val_loader))
image = batch["images"][0]
target_class = 0

heatmap = make_heatmap(model, image, target_class, device)
show_heatmap(image, heatmap)